# MS-CXR-T Temporal Progression Evaluation (K=1, K=2, K=4)

Evaluates the BioViL-T multi-prior checkpoints on the MS-CXR-T temporal
image-classification task using:

* **Linear probe** over the frozen fused image embedding (`img_global`, 128-d), with `GroupKFold` by `subject_id` (no patient leakage).
* **Zero-shot text prompts** (BioViL-T style): cosine similarity of `img_global` to per-class text prompts.

Each MS-CXR-T row provides a (current, prior, pathology, progression) label. We feed the model `current + K priors` where `prior_0` = the labeled prior and `prior_1..` = older MIMIC frontal priors (newest-first), so the label always refers to the current↔prior_0 transition, and K just adds older context.

### What's on Google Drive

The notebook reads from `/content/drive/MyDrive/output_share/` (shortcut). Expected layout:

```
output_share/
  mscxrt.zip                       # MIMIC-layout JPGs (pXX/pSUBJECT/sSTUDY/*.jpg)
  mscxrt_eval_imagelevel.csv       # built by biovilt/build_mscxrt_eval.py
  subset_checkpoints_4_3.zip       # K=4 trained checkpoint (contains best.pt)
  gcp-k1/
    subset_checkpoints_1.zip       # K=1 trained checkpoint (contains best.pt)
  k2/
    subset_checkpoints.zip         # K=2 trained checkpoint (contains best.pt)
```

Heavy files (mscxrt.zip, checkpoint zips) are **extracted to the Colab VM** under `/content/eval_work/` so inference doesn't hit Drive on every image read. The eval CSV is read directly from Drive (tiny). Metrics from each K are cached to disk so you can run **section 5 (K=2) alone** after a prior full run without re-running K=1/K=4 inference. The notebook clones the repo and reuses `BioViLTDataset` and `TempCXR` unchanged.

## 1. Install deps + clone repo

In [ ]:
%pip install -q transformers hi-ml-multimodal scikit-learn matplotlib pillow

In [ ]:
import os, sys, subprocess

REPO_URL  = 'https://github.com/nprakash1/cxr-temporal-multiprior.git'
REPO_PATH = '/content/cxr-temporal'
REPO_BRANCH = 'gcp-release'

if not os.path.isdir(REPO_PATH):
    subprocess.check_call(['git','clone','--depth','1','-b', REPO_BRANCH, REPO_URL, REPO_PATH])
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, 'biovilt'))
print('repo at:', REPO_PATH)

## 2. Mount Drive + shared paths (images + eval CSV)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, time
from pathlib import Path

# ---- Drive paths (the output_share shortcut you added to MyDrive) ----
OUTPUT_SHARE = '/content/drive/MyDrive/output_share'
ZIP_MSCXRT   = os.path.join(OUTPUT_SHARE, 'mscxrt.zip')
ZIP_K1       = os.path.join(OUTPUT_SHARE, 'gcp-k1', 'subset_checkpoints_1.zip')
ZIP_K2       = os.path.join(OUTPUT_SHARE, 'k2', 'subset_checkpoints.zip')
ZIP_K4       = os.path.join(OUTPUT_SHARE, 'subset_checkpoints_4_3.zip')
EVAL_CSV     = os.path.join(OUTPUT_SHARE, 'mscxrt_eval_imagelevel.csv')

assert os.path.exists(OUTPUT_SHARE), f'output_share not found — open {OUTPUT_SHARE} in the Files pane and verify the shortcut'
for label, p in [('mscxrt.zip', ZIP_MSCXRT), ('K=1 zip', ZIP_K1),
                  ('K=2 zip', ZIP_K2), ('K=4 zip', ZIP_K4),
                  ('eval csv', EVAL_CSV)]:
    print(f'{label:11s}: {p}   exists={os.path.exists(p)}')

# ---- Extract shared images to fast local scratch (checkpoint zips unzipped per-section) ----
LOCAL = Path('/content/eval_work'); LOCAL.mkdir(exist_ok=True)
CACHE_DIR = LOCAL / 'cached_metrics'; CACHE_DIR.mkdir(exist_ok=True)
MSCXRT_DIR = LOCAL / 'mscxrt'

def _unzip(zip_path, dest_dir, sentinel_glob):
    dest_dir = Path(dest_dir)
    if dest_dir.exists() and list(dest_dir.rglob(sentinel_glob)):
        print(f'  [skip] {dest_dir} already populated')
        return
    dest_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(f'  unzip {zip_path}  ->  {dest_dir}   ...', end='', flush=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest_dir)
    print(f' done in {time.time()-t0:.1f}s')

print('\n[unzip] images ...')
_unzip(ZIP_MSCXRT, MSCXRT_DIR, sentinel_glob='*.jpg')

def _find_mimic_root(base):
    base = Path(base)
    for cand in [base] + [p.parent for p in base.rglob('p1[0-9]') if p.is_dir()][:1]:
        if any((cand / f'p1{i}').is_dir() for i in range(10)):
            return str(cand)
    raise FileNotFoundError(f'Could not find MIMIC pXX layout under {base}')
IMG_ROOT = _find_mimic_root(MSCXRT_DIR)

def _find_best_pt(base):
    base = Path(base)
    cands = sorted(base.rglob('best.pt'))
    if cands: return str(cands[0])
    cands = sorted(base.rglob('*.pt'))
    if not cands:
        raise FileNotFoundError(f'No .pt under {base}')
    cands.sort(key=lambda p: ('best' not in p.name.lower(), -p.stat().st_size))
    return str(cands[0])

print('\nResolved shared paths:')
for label, p in [('image root', IMG_ROOT), ('eval csv', EVAL_CSV)]:
    print(f'  {label:11s}: {p}   exists={os.path.exists(p)}')

## 3. Imports + helpers

In [ ]:
import json, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader

from biovilt.dataset import BioViLTDataset, biovilt_collate_fn
from biovilt.tempcxr.modules.tempcxr_model import TempCXR

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| gpu:', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'n/a')

In [ ]:
def build_loader(k_max, batch_size, num_workers=2):
    ds = BioViLTDataset(
        csv_path=EVAL_CSV, image_root=IMG_ROOT,
        split='test', train=False, k_max=k_max,
    )
    dl = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        collate_fn=biovilt_collate_fn,
    )
    return ds, dl

def load_model(ckpt_path, k_max, mode='biovilt'):
    """Build TempCXR(mode, K_max) and load the trained weights.
    Handles raw state_dicts, dicts with 'model_state_dict'/'state_dict',
    and DDP 'module.' prefixes. Uses strict=False since the text encoder
    HuggingFace blobs are reloaded from upstream."""
    model = TempCXR(mode=mode, K_max=k_max).to(DEVICE)
    sd = torch.load(ckpt_path, map_location='cpu')
    if isinstance(sd, dict):
        for key in ('model', 'model_state_dict', 'state_dict'):
            if key in sd and isinstance(sd[key], dict):
                sd = sd[key]; break
    sd = {(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f'   loaded {ckpt_path}\n   missing={len(missing)}  unexpected={len(unexpected)}')
    model.eval()
    return model

@torch.no_grad()
def extract_embeddings(model, dl):
    feats = []
    for i, batch in enumerate(dl):
        curr   = batch['current_image'].to(DEVICE, non_blocking=True)
        priors = batch['prior_images']
        mask   = batch['prior_mask']
        if priors is not None: priors = priors.to(DEVICE, non_blocking=True)
        if mask   is not None: mask   = mask.to(DEVICE, non_blocking=True)
        img_global, _ = model.image_encoder(curr, priors, mask)
        feats.append(img_global.cpu().numpy())
        if (i+1) % 20 == 0:
            print(f'  batch {i+1}/{len(dl)}')
    return np.concatenate(feats, axis=0)

@torch.no_grad()
def embed_prompts(model, prompts):
    """Prompt-ensemble per class.

    Args:
        prompts: dict pathology -> list[list[str]] of length 3 (one inner
                 list of paraphrases per MS-CXR-T class, in the order
                 [improving, stable, worsening]).
    Returns:
        dict pathology -> np.ndarray (3, D), unit-normalized.
    """
    out = {}
    for p, cls_prompts in prompts.items():
        rows = []
        for class_prompts in cls_prompts:
            txt_global, _, _ = model.text_encoder.forward_contrastive(class_prompts)
            mean_emb = txt_global.mean(dim=0, keepdim=True)
            mean_emb = torch.nn.functional.normalize(mean_emb, dim=-1)
            rows.append(mean_emb)
        out[p] = torch.cat(rows, dim=0).cpu().numpy()   # (3, D)
    return out

def run_checkpoint(ckpt_path, k_max, batch_size):
    print(f'=== {ckpt_path}  (k_max={k_max}, bs={batch_size}) ===')
    ds, dl = build_loader(k_max=k_max, batch_size=batch_size)
    model = load_model(ckpt_path, k_max=k_max)
    img_emb = extract_embeddings(model, dl)
    txt_emb = embed_prompts(model, prompts_for_pathology)
    print('  image embeddings:', img_emb.shape)
    print('  prompt embeddings per pathology:',
          {p: v.shape for p, v in txt_emb.items()})
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return img_emb, txt_emb

## 4. Read labels + prompt templates

In [ ]:
df = pd.read_csv(EVAL_CSV)
print('rows:', len(df))
print(df[['progression_label','pathology','num_priors','num_older_priors']].head())

CLASSES = ['improving', 'stable', 'worsening']
y           = np.array([CLASSES.index(s) for s in df['progression_label']])
pathologies = df['pathology'].values
subjects    = df['subject_id'].values
pathologies_unique = sorted(np.unique(pathologies).tolist())
print('\nlabel counts:'); print(pd.Series(df['progression_label']).value_counts())
print('\npathology counts:'); print(pd.Series(pathologies).value_counts())
print('\nunique subjects:', len(np.unique(subjects)))

In [ ]:
# Per-pathology prompt ensemble. CLASSES order is [improving, stable, worsening].
_CLASS_PARAPHRASES = {
    'improving': [
        'the {p} is improving',
        'improvement in {p}',
        'decreased {p}',
        '{p} has improved',
    ],
    'stable': [
        'the {p} is stable',
        'no change in {p}',
        '{p} is unchanged',
        'stable {p}',
    ],
    'worsening': [
        'the {p} is worsening',
        'worsening {p}',
        'increased {p}',
        '{p} has worsened',
    ],
}
prompts_for_pathology = {
    p: [[tpl.format(p=p) for tpl in _CLASS_PARAPHRASES[c]] for c in CLASSES]
    for p in pathologies_unique
}
print('prompts (showing one pathology):')
for c, plist in zip(CLASSES, prompts_for_pathology[pathologies_unique[0]]):
    print(f'  {c:10s}:', plist)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score

def probe_eval(X, y, groups, pathologies_arr, n_splits=5, name=''):
    gkf = GroupKFold(n_splits=n_splits)
    accs, f1s = [], []
    by_path = {}
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups)):
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=2000, multi_class='multinomial', C=1.0)
        clf.fit(sc.transform(X[tr]), y[tr])
        pred = clf.predict(sc.transform(X[te]))
        accs.append(accuracy_score(y[te], pred))
        f1s.append(f1_score(y[te], pred, average='macro'))
        for p in np.unique(pathologies_arr):
            m = pathologies_arr[te] == p
            if m.sum() == 0: continue
            by_path.setdefault(p, []).append(
                f1_score(y[te][m], pred[m], average='macro', zero_division=0)
            )
    print(f'[{name}]  acc = {np.mean(accs):.3f} ± {np.std(accs):.3f}   '
          f'macro-F1 = {np.mean(f1s):.3f} ± {np.std(f1s):.3f}')
    for p in sorted(by_path):
        v = by_path[p]
        print(f'   {p:18s}: macro-F1 {np.mean(v):.3f} ± {np.std(v):.3f}  (n_folds={len(v)})')
    return {
        'condition': name,
        'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)),
        'f1_mean':  float(np.mean(f1s)),  'f1_std':  float(np.std(f1s)),
        'by_path': {p: (float(np.mean(v)), float(np.std(v))) for p, v in by_path.items()},
    }

def zeroshot_eval(img_emb, prompt_emb, pathologies_arr, y, name=''):
    img = img_emb / (np.linalg.norm(img_emb, axis=1, keepdims=True) + 1e-9)
    pred = np.empty(len(y), dtype=int)
    for i, p in enumerate(pathologies_arr):
        sims = img[i] @ prompt_emb[p].T
        pred[i] = int(np.argmax(sims))
    acc = accuracy_score(y, pred)
    f1  = f1_score(y, pred, average='macro')
    print(f'[{name}]  acc = {acc:.3f}   macro-F1 = {f1:.3f}')
    by_path = {}
    for p in np.unique(pathologies_arr):
        m = pathologies_arr == p
        by_path[p] = f1_score(y[m], pred[m], average='macro', zero_division=0)
        print(f'   {p:18s}: macro-F1 {by_path[p]:.3f}  (n={int(m.sum())})')
    return {'condition': name, 'acc': float(acc), 'f1': float(f1),
            'by_path': by_path}

def _metrics_to_json(probe, zs):
    return {
        'probe': {
            **probe,
            'by_path': {p: list(v) for p, v in probe['by_path'].items()},
        },
        'zs': zs,
    }

def save_metrics(k, probe, zs):
    path = CACHE_DIR / f'metrics_k{k}.json'
    with open(path, 'w') as f:
        json.dump(_metrics_to_json(probe, zs), f, indent=2)
    print(f'cached metrics -> {path}')

def load_metrics(k):
    path = CACHE_DIR / f'metrics_k{k}.json'
    if not path.exists():
        return None
    with open(path) as f:
        d = json.load(f)
    d['probe']['by_path'] = {p: tuple(v) for p, v in d['probe']['by_path'].items()}
    return d['probe'], d['zs']

def get_metrics(k):
    probe_name, zs_name = f'probe_k{k}', f'zs_k{k}'
    if probe_name in globals() and zs_name in globals():
        return globals()[probe_name], globals()[zs_name]
    loaded = load_metrics(k)
    if loaded is not None:
        print(f'loaded cached metrics for K={k}')
        return loaded
    return None, None

## 5. K=2 only — download, inference, and eval

**To add K=2 without re-running K=1/K=4:** run sections 1–5 only, then jump to section 7 (summary). K=1/K=4 metrics are loaded from `/content/eval_work/cached_metrics/` if you ran section 6 in a prior session on the same VM.

In [ ]:
K2_DIR = LOCAL / 'ckpt_k2'
print('[unzip] K=2 checkpoint ...')
_unzip(ZIP_K2, K2_DIR, sentinel_glob='*.pt')
CKPT_K2 = _find_best_pt(K2_DIR)
print(f'K=2 ckpt: {CKPT_K2}   exists={os.path.exists(CKPT_K2)}')

In [ ]:
emb_k2, prompt_emb_k2 = run_checkpoint(CKPT_K2, k_max=2, batch_size=6)
probe_k2 = probe_eval(emb_k2, y, subjects, pathologies, name='probe / K=2 model, K=2 input')
zs_k2 = zeroshot_eval(emb_k2, prompt_emb_k2, pathologies, y, name='zero-shot / K=2 model')
save_metrics(2, probe_k2, zs_k2)

## 6. K=1 and K=4 — download, inference, and eval

Skip this section if you only need K=2 and already have K=1/K=4 metrics cached from a prior run.

In [ ]:
K1_DIR = LOCAL / 'ckpt_k1'
K4_DIR = LOCAL / 'ckpt_k4'
print('[unzip] K=1 checkpoint ...')
_unzip(ZIP_K1, K1_DIR, sentinel_glob='*.pt')
print('[unzip] K=4 checkpoint ...')
_unzip(ZIP_K4, K4_DIR, sentinel_glob='*.pt')
CKPT_K1 = _find_best_pt(K1_DIR)
CKPT_K4 = _find_best_pt(K4_DIR)
for label, p in [('K=1 ckpt', CKPT_K1), ('K=4 ckpt', CKPT_K4)]:
    print(f'  {label:11s}: {p}   exists={os.path.exists(p)}')

In [ ]:
emb_k1, prompt_emb_k1 = run_checkpoint(CKPT_K1, k_max=1, batch_size=8)
emb_k4, prompt_emb_k4 = run_checkpoint(CKPT_K4, k_max=4, batch_size=4)

In [ ]:
probe_k1 = probe_eval(emb_k1, y, subjects, pathologies, name='probe / K=1 model, K=1 input')
probe_k4 = probe_eval(emb_k4, y, subjects, pathologies, name='probe / K=4 model, K=4 input')
zs_k1 = zeroshot_eval(emb_k1, prompt_emb_k1, pathologies, y, name='zero-shot / K=1 model')
zs_k4 = zeroshot_eval(emb_k4, prompt_emb_k4, pathologies, y, name='zero-shot / K=4 model')
save_metrics(1, probe_k1, zs_k1)
save_metrics(4, probe_k4, zs_k4)

## 7. Combined summary

Loads in-memory results when available, otherwise falls back to cached metrics on disk.

In [ ]:
ALL_K = [1, 2, 4]
results = {}
for k in ALL_K:
    probe, zs = get_metrics(k)
    if probe is not None and zs is not None:
        results[k] = {'probe': probe, 'zs': zs}
    else:
        print(f'[skip] no metrics for K={k} — run section {5 if k==2 else 6} first')

summary_rows = []
for k in ALL_K:
    if k not in results:
        continue
    probe, zs = results[k]['probe'], results[k]['zs']
    summary_rows.append({
        'condition': probe['condition'],
        'acc': f"{probe['acc_mean']:.3f} ± {probe['acc_std']:.3f}",
        'macro_F1': f"{probe['f1_mean']:.3f} ± {probe['f1_std']:.3f}",
    })
    summary_rows.append({
        'condition': zs['condition'],
        'acc': f"{zs['acc']:.3f}",
        'macro_F1': f"{zs['f1']:.3f}",
    })
print(pd.DataFrame(summary_rows).to_string(index=False))

rows = []
for p in pathologies_unique:
    row = {'pathology': p}
    for k in ALL_K:
        if k not in results:
            continue
        probe, zs = results[k]['probe'], results[k]['zs']
        m, s = probe['by_path'].get(p, (0, 0))
        row[f'probe_K={k}'] = f'{m:.3f} ± {s:.3f}'
        row[f'zs_K={k}'] = f"{zs['by_path'].get(p, 0):.3f}"
    rows.append(row)
print('\nPer-pathology macro-F1:')
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

names, f1_vals, f1_errs, colors = [], [], [], []
palette = {1: '#1f77b4', 2: '#2ca02c', 4: '#9467bd'}
for k in ALL_K:
    if k not in results:
        continue
    probe, zs = results[k]['probe'], results[k]['zs']
    names += [f'probe K={k}', f'zero-shot K={k}']
    f1_vals += [probe['f1_mean'], zs['f1']]
    f1_errs += [probe['f1_std'], 0]
    colors += [palette[k], palette[k]]

if names:
    fig, ax = plt.subplots(figsize=(max(8, len(names) * 1.1), 4.5))
    bars = ax.bar(names, f1_vals, yerr=f1_errs, capsize=4, color=colors, alpha=0.85)
    ax.set_ylabel('macro-F1')
    ax.set_title('MS-CXR-T temporal progression: K=1, K=2, K=4')
    for b, v in zip(bars, f1_vals):
        ax.text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}',
                ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, max(f1_vals)*1.25 + 0.05)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout(); plt.show()
else:
    print('No results to plot — run at least one checkpoint section first.')